# trio-retina — camera → restricted-zone alert → webhook

**What you'll see:** a person walks into a restricted zone, Retina fires a `zone.enter` event, and a sink "POSTs" it to your backend — the deploy story, running with no camera and no network.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machinefi/trio-retina/blob/main/notebooks/02_camera_to_webhook.ipynb)


Install from PyPI — pure Python + numpy.

In [ ]:
%pip install trio-retina

## A restricted zone + a synthetic feed

The zone uses normalized `0..1` coords so the same rule works at any resolution. The detector is a stub that walks one `person` through the zone.

In [ ]:
import numpy as np
from retina import Retina, Zone, ZoneRule, WebhookSink
from retina.detect import Detection


class ScriptedDetector:
    """Stub 'model': walks one person left-to-right, one step per frame."""

    def __init__(self):
        self.f = 0

    def __call__(self, frame: np.ndarray) -> list[Detection]:
        x = self.f * 6
        self.f += 1
        return [Detection("person", (x - 10, 40, x + 10, 60), 0.9)] if x <= 96 else []


restricted = Zone("restricted", [(0.4, 0), (0.6, 0), (0.6, 1), (0.4, 1)], normalized=True)
frames = [(np.zeros((100, 100, 3), np.uint8), float(i)) for i in range(18)]

## A sink that prints instead of POSTing

A sink is any `callable(event) -> None`. In production you use `WebhookSink("https://your-endpoint")` (stdlib `urllib`, no `requests` dep), which POSTs each event as JSON. To keep this notebook offline we use a thin fake that prints `POST →` — same shape, no network.

In [ ]:
class PrintSink:
    """Stand-in for WebhookSink: print instead of POST (keeps the notebook offline)."""

    def __call__(self, event) -> None:
        print("POST -> https://your-endpoint  ", event.to_json())


# In a real deployment, swap the line above for:
#     sink = WebhookSink("https://your-endpoint")   # POSTs each event as JSON
sink = PrintSink()

## Run it

`Retina` pushes every fired event to the sink (and we also print the yielded event). The two lines a real deployment changes are marked **SWAP** in the comments — point the source at your camera, point the sink at your endpoint, and the rule + event stream are byte-for-byte identical.

In [ ]:
cam = Retina(
    source_id="cam_01",
    detector=ScriptedDetector(),                       # SWAP: YoloDetector("yolo11n.pt", classes={"person"})
    rules=[ZoneRule(restricted, classes={"person"})],
    sinks=[sink],                                      # SWAP: [WebhookSink("https://your-endpoint")]
)

# SWAP the source too: video_frames("rtsp://CAM", live=True) for a live camera.
# from retina.sources import video_frames
for event in cam.run(frames):
    print("yielded:", event.to_json())